In [0]:
dbutils.fs.mkdirs("/FileStoreShivam/hr/raw")

##Bronze layer

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Configuration based on your setup
RAW_PATH = "/FileStore/tables/FileStoreShivam/hr/raw/"
CATALOG = "shivam_project"
SCHEMA = "bronze"

# List of tables to process
tables = ["employees", "salaries", "departments"]

for table in tables:
    # 1. Read the raw CSV and explicitly select the hidden _metadata column
    # This replaces input_file_name() to satisfy Unity Catalog requirements
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(f"{RAW_PATH}{table}.csv")
          .select("*", "_metadata.file_path")) # Accessing UC-supported metadata
    
    # 2. Add metadata columns using the correct UC syntax
    # Mapping file_path to _source_file as per Task 2 requirements
    df_bronze = (df.withColumn("_ingested_at", current_timestamp())
                   .withColumnRenamed("file_path", "_source_file"))
    
    # 3. Save to the Bronze layer
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    
    print(f"Success: Table {table} ingested into {CATALOG}.{SCHEMA}")

In [0]:
%sql
ALTER TABLE shivam_project.bronze.employees 
ADD CONSTRAINT valid_employment_type 
CHECK (employment_type IN ('full_time', 'part_time', 'contract'));

##silver layer

In [0]:
(spark.readStream.format("cloudFiles").option("cloudFiles.schemaLocation", "/FileStore/schemaLocation/")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .load("/FileStore/tables/FileStoreShivam/hr/raw/")
  .filter("_metadata.file_path LIKE '%employees_update.csv%'")
  .withColumn("_ingested_at", current_timestamp())
  .withColumn("_source_file", col("_metadata.file_path"))
  .writeStream
  .option("checkpointLocation", "/FileStore/checkpoints/bronze_updates")
  .trigger(availableNow=True)
  .toTable("shivam_project.bronze.employees_updates"))

In [0]:
(spark.readStream.format("cloudFiles")  .option("cloudFiles.schemaLocation", "/FileStore/schemaLocation/employees_updates")

  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .load("/FileStore/tables/FileStoreShivam/hr/raw/")
  .filter("_metadata.file_path LIKE '%employees_update.csv%'")
  .withColumn("_ingested_at", current_timestamp())
  .withColumn("_source_file", col("_metadata.file_path"))
  .writeStream
  .option("checkpointLocation", "/FileStore/checkpoints/bronze_updates")
  .trigger(availableNow=True)
  .toTable("shivam_project.bronze.employees_updates"))

In [0]:
from pyspark.sql.functions import col, current_date, datediff, year, rank
from pyspark.sql.window import Window

# 1. Read Bronze Employees and calculate Tenure
employees_df = spark.read.table("shivam_project.bronze.employees") \
    .withColumn("hire_date", col("hire_date").cast("date")) \
    .withColumn("tenure_years", (datediff(current_date(), col("hire_date")) / 365.25))

# 2. Get latest salary using Window function (Rank 1 = newest)
salary_window = Window.partitionBy("employee_id").orderBy(col("effective_date").desc())
latest_salaries = spark.read.table("shivam_project.bronze.salaries") \
    .withColumn("rn", rank().over(salary_window)) \
    .filter("rn = 1") \
    .withColumn("total_monthly_compensation", col("base_salary") + (col("base_salary") * col("bonus_pct")))

# 3. Join with Departments and Save to Silver
departments_df = spark.read.table("shivam_project.bronze.departments")

employees_enriched = employees_df.join(latest_salaries, "employee_id", "left") \
    .join(departments_df, "department_id", "left") \
    .select("employee_id", "full_name", "email", "department_name", "location", 
            "job_title", "hire_date", "tenure_years", "total_monthly_compensation")

employees_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("shivam_project.silver.employees_enriched")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS shivam_project.silver.employees_history (
    employee_id STRING,
    full_name STRING,
    job_title STRING,
    is_current BOOLEAN,
    effective_start_date DATE,
    effective_end_date DATE
)
TBLPROPERTIES (delta.enableChangeDataFeed = true); -- Requirement: Enable CDF

In [0]:
# This logic checks if the job_title in Bronze differs from the current record in Silver
# If it differs, it "expires" the old record and inserts a new one.

from pyspark.sql.functions import current_date, lit

# Define updates from your Bronze updates table
updates_df = spark.read.table("shivam_project.bronze.employees_updates")

# Register as a temporary view to use in SQL MERGE
updates_df.createOrReplaceTempView("v_updates")

spark.sql("""
  MERGE INTO shivam_project.silver.employees_history AS target
  USING (
    -- Identify the records that need to be updated or inserted
    SELECT employee_id as mergeKey, * FROM v_updates
    UNION ALL
    SELECT NULL as mergeKey, u.* FROM v_updates u
    JOIN shivam_project.silver.employees_history t
    ON u.employee_id = t.employee_id
    WHERE t.is_current = true AND u.job_title <> t.job_title
  ) AS source
  ON target.employee_id = source.mergeKey
  WHEN MATCHED AND target.is_current = true AND target.job_title <> source.job_title THEN
    UPDATE SET target.is_current = false, target.effective_end_date = current_date()
  WHEN NOT MATCHED THEN
    INSERT (employee_id, full_name, job_title, is_current, effective_start_date, effective_end_date)
    VALUES (source.employee_id, source.full_name, source.job_title, true, current_date(), NULL)
""")

In [0]:
%sql
-- Optimize the enriched table for faster joins and filters
OPTIMIZE shivam_project.silver.employees_enriched 
ZORDER BY (department_name, hire_date);

-- Check if the Change Data Feed is working
SELECT * FROM table_changes('shivam_project.silver.employees_history', 1);

GOLD LAYER PROCESS

##Gold layer

In [0]:
%sql
CREATE OR REPLACE VIEW shivam_project.gold.dept_payroll_summary AS
SELECT 
    department_name, 
    location, 
    COUNT(employee_id) AS head_count,
    ROUND(SUM(total_monthly_compensation), 2) AS total_monthly_payroll,
    ROUND(AVG(total_monthly_compensation), 2) AS avg_compensation
FROM shivam_project.silver.employees_enriched
GROUP BY department_name, location;

##Create the Headcount vs. Budget View

In [0]:
%sql
CREATE OR REPLACE VIEW shivam_project.gold.headcount_vs_budget AS
SELECT 
    d.department_name,
    COUNT(e.employee_id) AS actual_headcount,
    d.head_count_budget,
    (d.head_count_budget - COUNT(e.employee_id)) AS remaining_budget
FROM shivam_project.silver.employees_enriched e
JOIN shivam_project.bronze.departments d ON e.department_name = d.department_name
GROUP BY d.department_name, d.head_count_budget;

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_timestamp

# 1. Define the schema to match the salaries.csv specification 
salary_schema = "employee_id STRING, base_salary DOUBLE, bonus_pct DOUBLE, effective_date STRING"

# 2. Read the streaming data using Auto Loader [cite: 51]
streaming_salaries = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(salary_schema)
    .load("/FileStore/tables/FileStoreShivam/hr/raw/"))

# 3. Filter for the specific batch and apply 1-day Watermark [cite: 52]
processed_stream = (streaming_salaries
    .filter("_metadata.file_path LIKE '%salaries_batch2.csv%'")
    # Explicitly specify the date format to avoid casting errors from other columns
    .withColumn("effective_date", to_timestamp(col("effective_date"), "yyyy-MM-dd"))
    .withWatermark("effective_date", "1 day") 
    .withColumn("_ingested_at", current_timestamp()))

# 4. CRITICAL: Clear the old checkpoint before restarting to fix the 'malformed' state
dbutils.fs.rm("/FileStore/checkpoints/gold_salary_updates", True)

# 5. Write the stream to the Gold table 
query = (processed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/FileStore/checkpoints/gold_salary_updates")
    .trigger(availableNow=True)
    .toTable("shivam_project.gold.live_salary_updates"))